# A09. Long-Tail 시뮬레이션

**3가지 Long-Tail 현상을 시뮬레이션으로 체험한다.**

| 시뮬레이션 | 주제 | 핵심 개념 |
|:---:|:---|:---|
| 1 | 유튜브 조회수 | Power-law 분포 |
| 2 | 부의 불평등 | Pareto 80/20 법칙 |
| 3 | 네트워크 성장 | Barabasi-Albert 모델 |

---
## 양춘백설(陽春白雪) — Long Tail의 고전적 비유

> 고대 중국 초(楚)나라의 노래 이야기가 있다.
>
> 어느 가수가 수도에서 노래를 불렀다.  
> 처음에 **하리파인(下里巴人)** — 쉽고 대중적인 노래를 부르자, **수천 명**이 따라 불렀다.  
> 다음에 **양아백설(陽阿薤露)** — 조금 어려운 노래를 부르자, **수백 명**만 따라 불렀다.  
> 마지막에 **양춘백설(陽春白雪)** — 가장 고급스러운 노래를 부르자, **겨우 수십 명**만 따라 불렀다.
>
> — *송옥(宋玉), 「대초왕문(對楚王問)」*

### 이것이 바로 Long-Tail 현상이다

- **Head (머리)**: 소수의 히트곡을 수천 명이 따라 부른다 → 소수가 대부분의 관심을 독점
- **Long Tail (긴 꼬리)**: 고급 음악은 소수만 감상한다 → 다수의 비인기 콘텐츠가 긴 꼬리를 형성

2000년이 넘은 이야기지만, 유튜브 조회수, 부의 분포, 인터넷 링크 구조까지 — **동일한 패턴이 반복**된다.

$$
P(X > x) \propto x^{-\alpha}
$$

멱법칙(Power Law): 값이 커질수록 빈도가 급격히 줄어들지만, 완전히 0이 되지는 않는다.

In [ ]:
# === 공통 설정 ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy import stats

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

# 재현성
np.random.seed(42)

print("라이브러리 로드 완료")

---
## 시뮬레이션 1: 유튜브 조회수 Long-Tail

유튜브에는 수억 개의 동영상이 있지만, 대부분의 조회수는 극소수 영상에 집중된다.  
이를 **Power-law 분포**로 시뮬레이션해 보자.

$$
\text{조회수} \sim \text{Pareto}(\alpha) \quad \Rightarrow \quad f(x) = \frac{\alpha}{x^{\alpha+1}}, \quad x \geq 1
$$

In [ ]:
# 100,000개 동영상 조회수 생성 (Pareto 분포)
n_videos = 100_000
alpha = 1.5  # 형상 모수 (작을수록 꼬리가 길다)

# scipy.stats.pareto로 생성 후 스케일링 (최소 조회수 100)
views = (stats.pareto.rvs(b=alpha, size=n_videos) * 100).astype(int)
views = np.clip(views, 100, None)  # 최소 100회

print(f"총 동영상 수: {n_videos:,}")
print(f"총 조회수: {views.sum():,}")
print(f"평균 조회수: {views.mean():,.0f}")
print(f"중앙값 조회수: {np.median(views):,.0f}")
print(f"최대 조회수: {views.max():,}")
print(f"최소 조회수: {views.min():,}")

In [ ]:
# 상위 1% 동영상이 전체 조회수의 몇 %를 차지하는가?
sorted_views = np.sort(views)[::-1]  # 내림차순 정렬
top_1_pct = int(n_videos * 0.01)
top_5_pct = int(n_videos * 0.05)
top_10_pct = int(n_videos * 0.10)

share_1 = sorted_views[:top_1_pct].sum() / views.sum() * 100
share_5 = sorted_views[:top_5_pct].sum() / views.sum() * 100
share_10 = sorted_views[:top_10_pct].sum() / views.sum() * 100

print("=" * 50)
print("Long-Tail 집중도 분석")
print("=" * 50)
print(f"상위  1% ({top_1_pct:,}개) → 전체 조회수의 {share_1:.1f}% 차지")
print(f"상위  5% ({top_5_pct:,}개) → 전체 조회수의 {share_5:.1f}% 차지")
print(f"상위 10% ({top_10_pct:,}개) → 전체 조회수의 {share_10:.1f}% 차지")
print(f"\n나머지 90% ({n_videos - top_10_pct:,}개)는 전체의 {100 - share_10:.1f}%만 차지")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) 로그 스케일 히스토그램
ax = axes[0]
log_bins = np.logspace(np.log10(views.min()), np.log10(views.max()), 50)
ax.hist(views, bins=log_bins, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('조회수 (log scale)', fontsize=12)
ax.set_ylabel('동영상 수 (log scale)', fontsize=12)
ax.set_title('유튜브 조회수 분포 (Log-Log)', fontsize=14, fontweight='bold')

# (2) Head vs Long Tail 영역 시각화
ax = axes[1]
cumulative_share = np.cumsum(sorted_views) / views.sum() * 100
x_rank = np.arange(1, n_videos + 1) / n_videos * 100

ax.fill_between(x_rank[:top_10_pct], cumulative_share[:top_10_pct],
                color='#e74c3c', alpha=0.6, label=f'Head (상위 10%) = {share_10:.0f}%')
ax.fill_between(x_rank[top_10_pct:], cumulative_share[top_10_pct - 1],
                cumulative_share[top_10_pct:],
                color='#3498db', alpha=0.4, label=f'Long Tail (나머지 90%) = {100-share_10:.0f}%')
ax.plot(x_rank, cumulative_share, color='black', linewidth=1.5)
ax.set_xlabel('동영상 순위 (%)', fontsize=12)
ax.set_ylabel('누적 조회수 비율 (%)', fontsize=12)
ax.set_title('Head vs Long Tail', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='center right')
ax.set_xlim(0, 100)
ax.set_ylim(0, 105)

# (3) 상위 100개 동영상의 조회수
ax = axes[2]
top_100 = sorted_views[:100]
colors = ['#e74c3c' if i < 10 else '#f39c12' if i < 50 else '#3498db' for i in range(100)]
ax.bar(range(100), top_100, color=colors, width=1.0)
ax.set_xlabel('순위', fontsize=12)
ax.set_ylabel('조회수', fontsize=12)
ax.set_title('상위 100개 동영상 조회수', fontsize=14, fontweight='bold')
ax.ticklabel_format(style='plain', axis='y')

plt.tight_layout()
plt.show()

### 해석

- **소수의 동영상이 대부분의 조회수를 독점**한다 (Head 영역)
- 나머지 대다수 동영상은 조회수가 매우 적다 (Long Tail 영역)
- Log-Log 그래프에서 **직선에 가까운 형태** → Power-law 분포의 특징
- 이것이 유튜브 알고리즘의 "승자 독식" 구조를 만든다

> **양춘백설의 교훈**: 고급 콘텐츠는 소수만 소비하지만, 그 소수가 매우 깊이 소비한다.  
> 반대로 대중적 콘텐츠는 많은 사람이 소비하지만, 깊이는 얕다.

---
## 시뮬레이션 2: 부의 불평등 (Pareto 80/20 법칙)

이탈리아 경제학자 파레토(Vilfredo Pareto)는 1896년에 발견했다:  
**"이탈리아 전체 토지의 80%를 인구의 20%가 소유한다."**

$$
X \sim \text{Pareto}(b=1.16, \text{scale}=x_m)
$$

- $b = 1.16$: 현실 세계의 부의 분포에 근접한 형상 모수
- Lorenz 곡선과 Gini 계수로 불평등 정도를 측정한다

In [ ]:
# 10,000명의 자산 생성
n_people = 10_000
b = 1.16  # Pareto 형상 모수
min_wealth = 10_000  # 최소 자산 (만원 단위)

wealth = stats.pareto.rvs(b=b, size=n_people) * min_wealth

print(f"총 인구: {n_people:,}명")
print(f"총 자산: {wealth.sum():,.0f} 만원")
print(f"평균 자산: {wealth.mean():,.0f} 만원")
print(f"중앙값 자산: {np.median(wealth):,.0f} 만원")
print(f"최대 자산: {wealth.max():,.0f} 만원")
print(f"평균/중앙값 비율: {wealth.mean()/np.median(wealth):.1f}배")

In [ ]:
# 80/20 법칙 검증
sorted_wealth = np.sort(wealth)[::-1]  # 내림차순
total_wealth = wealth.sum()

print("=" * 55)
print("Pareto 80/20 법칙 검증")
print("=" * 55)

for pct in [1, 5, 10, 20, 50]:
    n_top = int(n_people * pct / 100)
    share = sorted_wealth[:n_top].sum() / total_wealth * 100
    print(f"상위 {pct:2d}% ({n_top:>5,}명) → 전체 부의 {share:5.1f}% 보유")

print("\n하위 50%는 전체 부의", end=" ")
bottom_50 = sorted_wealth[n_people//2:].sum() / total_wealth * 100
print(f"{bottom_50:.1f}%만 보유")

In [ ]:
def lorenz_curve(data):
    """Lorenz 곡선 계산"""
    sorted_data = np.sort(data)
    n = len(data)
    cumulative_wealth = np.cumsum(sorted_data) / sorted_data.sum()
    cumulative_pop = np.arange(1, n + 1) / n
    return cumulative_pop, cumulative_wealth

def gini_coefficient(data):
    """Gini 계수 계산"""
    sorted_data = np.sort(data)
    n = len(data)
    index = np.arange(1, n + 1)
    return (2 * np.sum(index * sorted_data) / (n * np.sum(sorted_data))) - (n + 1) / n

gini = gini_coefficient(wealth)
print(f"Gini 계수: {gini:.4f}")
print(f"(0 = 완전 평등, 1 = 완전 불평등)")

if gini > 0.6:
    print("→ 매우 높은 불평등 (개발도상국 수준)")
elif gini > 0.4:
    print("→ 높은 불평등")
else:
    print("→ 중간 수준의 불평등")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# (1) 자산 분포 히스토그램
ax = axes[0]
log_bins = np.logspace(np.log10(wealth.min()), np.log10(wealth.max()), 50)
ax.hist(wealth, bins=log_bins, color='#2ecc71', edgecolor='white', alpha=0.8)
ax.set_xscale('log')
ax.axvline(wealth.mean(), color='red', linestyle='--', linewidth=2, label=f'평균: {wealth.mean():,.0f}')
ax.axvline(np.median(wealth), color='blue', linestyle='--', linewidth=2, label=f'중앙값: {np.median(wealth):,.0f}')
ax.set_xlabel('자산 (만원, log scale)', fontsize=12)
ax.set_ylabel('인구 수', fontsize=12)
ax.set_title('자산 분포 (Pareto)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)

# (2) Lorenz 곡선
ax = axes[1]
cum_pop, cum_wealth = lorenz_curve(wealth)
ax.plot(cum_pop, cum_wealth, color='#e74c3c', linewidth=2.5, label=f'Lorenz 곡선 (Gini={gini:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='완전 평등선')
ax.fill_between(cum_pop, cum_wealth, cum_pop, color='#e74c3c', alpha=0.15)
ax.set_xlabel('인구 누적 비율', fontsize=12)
ax.set_ylabel('부 누적 비율', fontsize=12)
ax.set_title('Lorenz 곡선 + Gini 계수', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# 80/20 포인트 표시
idx_80 = np.searchsorted(cum_pop, 0.80)
ax.plot(0.80, cum_wealth[idx_80], 'ro', markersize=10, zorder=5)
ax.annotate(f'80% 인구 → {cum_wealth[idx_80]*100:.0f}% 부',
            xy=(0.80, cum_wealth[idx_80]), xytext=(0.4, 0.6),
            fontsize=10, arrowprops=dict(arrowstyle='->', color='red'),
            color='red', fontweight='bold')

# (3) 상위 비율별 부 점유율
ax = axes[2]
pcts = [1, 5, 10, 20, 50]
shares = []
for p in pcts:
    n_top = int(n_people * p / 100)
    shares.append(sorted_wealth[:n_top].sum() / total_wealth * 100)

colors_bar = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db']
bars = ax.bar([f'상위 {p}%' for p in pcts], shares, color=colors_bar, edgecolor='white')
ax.axhline(80, color='red', linestyle=':', linewidth=1.5, alpha=0.7)
ax.text(4.5, 82, '80%', color='red', fontsize=10, ha='right')

for bar, s in zip(bars, shares):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{s:.1f}%', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('전체 부에서 차지하는 비율 (%)', fontsize=12)
ax.set_title('부의 집중도', fontsize=14, fontweight='bold')
ax.set_ylim(0, 105)

plt.tight_layout()
plt.show()

### 해석

- **Pareto 80/20 법칙**이 시뮬레이션에서도 확인된다
- Lorenz 곡선이 대각선에서 멀리 떨어질수록 불평등이 심하다
- **Gini 계수**는 Lorenz 곡선과 대각선 사이의 면적 비율이다
- 평균과 중앙값의 차이가 클수록 분포가 편향되어 있다

| 국가 | Gini 계수 |
|:---|:---:|
| 대한민국 | ~0.34 |
| 미국 | ~0.39 |
| 남아공 | ~0.63 |
| 슬로바키아 | ~0.23 |

---
## 시뮬레이션 3: Barabasi-Albert 네트워크

인터넷, SNS, 논문 인용 네트워크에서 발견되는 **무척도 네트워크(Scale-free Network)**

**핵심 메커니즘: 선호적 연결 (Preferential Attachment)**
- 새로운 노드가 추가될 때, 이미 **연결이 많은 노드에 더 잘 연결**된다
- "부익부 빈익빈" (The Rich Get Richer)

$$
P(\text{새 노드가 노드 } i \text{에 연결}) = \frac{k_i}{\sum_j k_j}
$$

여기서 $k_i$는 노드 $i$의 현재 연결 수(degree)이다.

In [ ]:
try:
    import networkx as nx
    HAS_NETWORKX = True
    print("networkx 버전:", nx.__version__)
except ImportError:
    HAS_NETWORKX = False
    print("networkx가 설치되어 있지 않습니다.")
    print("설치 명령: pip install networkx")
    print("\n→ networkx 없이도 수동 시뮬레이션으로 진행합니다.")

In [ ]:
if HAS_NETWORKX:
    # Barabasi-Albert 모델로 네트워크 생성
    n_nodes = 500
    m = 2  # 새 노드가 기존 노드에 연결하는 수
    G = nx.barabasi_albert_graph(n=n_nodes, m=m, seed=42)

    # Degree 분포 추출
    degrees = [d for n, d in G.degree()]
    degree_counts = pd.Series(degrees).value_counts().sort_index()

    print(f"노드 수: {G.number_of_nodes()}")
    print(f"엣지 수: {G.number_of_edges()}")
    print(f"평균 degree: {np.mean(degrees):.2f}")
    print(f"최대 degree: {max(degrees)} (Hub 노드)")
    print(f"최소 degree: {min(degrees)}")

    # 상위 허브 노드
    degree_dict = dict(G.degree())
    top_hubs = sorted(degree_dict.items(), key=lambda x: x[1], reverse=True)[:5]
    print(f"\n상위 5개 Hub 노드:")
    for node, deg in top_hubs:
        print(f"  노드 {node}: degree = {deg}")
else:
    # networkx 없이 수동 BA 모델 구현
    n_nodes = 500
    m = 2
    
    # 초기: 완전 그래프 (m+1 노드)
    degrees = np.zeros(n_nodes, dtype=int)
    edges = []
    # 초기 노드 연결
    for i in range(m + 1):
        for j in range(i + 1, m + 1):
            edges.append((i, j))
            degrees[i] += 1
            degrees[j] += 1
    
    # 순차적 노드 추가
    for new_node in range(m + 1, n_nodes):
        existing = degrees[:new_node]
        probs = existing / existing.sum()
        targets = np.random.choice(new_node, size=m, replace=False, p=probs)
        for t in targets:
            edges.append((new_node, t))
            degrees[new_node] += 1
            degrees[t] += 1
    
    degrees_list = degrees.tolist()
    degree_counts = pd.Series(degrees_list).value_counts().sort_index()
    
    print(f"노드 수: {n_nodes}")
    print(f"엣지 수: {len(edges)}")
    print(f"평균 degree: {np.mean(degrees_list):.2f}")
    print(f"최대 degree: {max(degrees_list)} (Hub 노드)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# (1) Degree 분포 (Log-Log)
ax = axes[0]
ax.scatter(degree_counts.index, degree_counts.values,
           color='#9b59b6', s=50, alpha=0.7, edgecolors='white')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Degree (연결 수)', fontsize=12)
ax.set_ylabel('빈도', fontsize=12)
ax.set_title('Degree 분포 (Log-Log)\n→ 직선 = Power Law', fontsize=14, fontweight='bold')

# (2) Degree 히스토그램
ax = axes[1]
if HAS_NETWORKX:
    deg_data = degrees
else:
    deg_data = degrees_list
ax.hist(deg_data, bins=30, color='#9b59b6', edgecolor='white', alpha=0.8)
ax.axvline(np.mean(deg_data), color='red', linestyle='--', linewidth=2,
           label=f'평균: {np.mean(deg_data):.1f}')
ax.set_xlabel('Degree', fontsize=12)
ax.set_ylabel('노드 수', fontsize=12)
ax.set_title('Degree 분포 히스토그램', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)

# (3) 네트워크 시각화
ax = axes[2]
if HAS_NETWORKX:
    # 노드 크기를 degree에 비례하게
    node_sizes = [d * 3 for d in degrees]
    node_colors = degrees
    
    pos = nx.spring_layout(G, seed=42, k=0.3)
    nx.draw_networkx_edges(G, pos, ax=ax, alpha=0.05, width=0.3)
    scatter = nx.draw_networkx_nodes(G, pos, ax=ax,
                                     node_size=node_sizes,
                                     node_color=node_colors,
                                     cmap='YlOrRd', alpha=0.8)
    
    # Hub 노드 라벨 표시
    hub_labels = {node: f'{node}\n(d={deg})' for node, deg in top_hubs[:3]}
    nx.draw_networkx_labels(G, pos, labels=hub_labels, ax=ax,
                            font_size=8, font_weight='bold', font_color='darkred')
    ax.set_title('BA 네트워크 시각화\n(큰 노드 = Hub)', fontsize=14, fontweight='bold')
else:
    # networkx 없을 때: degree 순위 그래프
    sorted_deg = sorted(degrees_list, reverse=True)
    ax.bar(range(len(sorted_deg)), sorted_deg, color='#9b59b6', alpha=0.7, width=1.0)
    ax.set_xlabel('노드 순위', fontsize=12)
    ax.set_ylabel('Degree', fontsize=12)
    ax.set_title('Degree 순위 (높은 순)', fontsize=14, fontweight='bold')

ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.show()

### 해석

- **Degree 분포가 Power Law**를 따른다 → Log-Log 플롯에서 직선
- 소수의 **Hub 노드**가 매우 많은 연결을 갖고, 대다수 노드는 적은 연결만 갖는다
- 이것이 **"부익부 빈익빈"(Preferential Attachment)**의 결과다

### 실제 사례

| 네트워크 | Hub 노드 | 일반 노드 |
|:---|:---|:---|
| 인터넷 | Google, Facebook | 개인 블로그 |
| 논문 인용 | 아인슈타인, 섀넌 | 대부분의 논문 |
| SNS 팔로워 | BTS, 오바마 | 일반 사용자 |
| 공항 노선 | 인천, 히드로 | 소도시 공항 |

---
## 종합 정리

### 3가지 시뮬레이션의 공통점

| | 유튜브 조회수 | 부의 분포 | 네트워크 |
|:---|:---|:---|:---|
| **Head** | 극소수 인기 영상 | 소수의 부자 | Hub 노드 |
| **Long Tail** | 대다수 비인기 영상 | 대다수의 서민 | 일반 노드 |
| **분포** | Power Law | Pareto | Power Law |
| **메커니즘** | 알고리즘 추천 | 복리 효과 | 선호적 연결 |

### 핵심 메시지

1. **정규분포가 아닌 세상**: 많은 현실 현상은 정규분포가 아닌 **멱법칙(Power Law)**을 따른다
2. **평균의 함정**: Long-tail 분포에서 평균은 대표값으로 적절하지 않다
3. **양춘백설의 현대적 의미**: 소수의 콘텐츠/노드/사람이 전체를 지배하는 구조는 시대를 초월한다